In [1]:
import os
os.chdir('/home/wiikai/factor')

import factorlab as lab
import pandas as pd
import numpy as np
import quool

industry_ret = quool.DatetimeTable("./data/industry-returns")

In [16]:
start='2014-01-01'
stop='2024-07-01'

shares = lab.quotes_day.read("circulation_a",  start=start,stop=stop)
price = lab.quotes_day.read("close",  start=start,stop=stop)
adjfactor = lab.quotes_day.read("adjfactor",  start=start,stop=stop)
size = shares * price * adjfactor
ind = lab.industry_info.read('first_industry_name', start=start,stop=stop)
ret = (price * adjfactor).pct_change(fill_method=None)

def get_weighted_industry_returns(market_cap, returns, industry):
    market_cap = market_cap.reindex(returns.index, axis=0).reindex(returns.columns, axis=1)
    industry = industry.reindex(returns.index, axis=0).reindex(returns.columns, axis=1)
    
    df = pd.DataFrame({
        'industry': industry.stack(),
        'market_cap': market_cap.stack(),
        'returns': returns.stack()
    })
    
    total_market_cap = df.groupby(['industry', df.index.get_level_values(0)])['market_cap'].sum()
    df['weighted_returns'] = df['market_cap'] * df['returns']
    weighted_returns = df.groupby(['industry', df.index.get_level_values(0)])['weighted_returns'].sum()
    res = (weighted_returns / total_market_cap).unstack(level=0)
    return res

data = get_weighted_industry_returns(size, ret, ind)

In [17]:
data.index.name = 'date'
for name in data.columns:
    if name not in industry_ret.columns:
        industry_ret.add({name: 'float32'})
industry_ret.update(data)

/home/wiikai/quool/quool/table.py:233: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  return pd.Grouper(level=0, freq=self._freq, sort=True)


In [20]:
industry_ret.read()

industry,交通运输,传媒,农林牧渔,医药,商贸零售,国防军工,基础化工,家电,建材,建筑,...,纺织服装,综合,计算机,轻工制造,通信,钢铁,银行,非银行金融,食品饮料,综合金融
date,,,,,,,,,,,,,,,,,,,,,
2014-01-02,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
2014-01-03,-0.016348,0.020507,-0.008784,-0.008446,0.002906,-0.018049,-0.012712,-0.028228,-0.021179,-0.017981,...,-0.018147,-0.020430,-0.015869,-0.011083,0.003453,-0.001821,-0.017783,-0.029090,-0.010662,NaN
2014-01-06,-0.030861,-0.034909,-0.041424,-0.024559,-0.036791,-0.040634,-0.035558,-0.022990,-0.034821,-0.035230,...,-0.032568,-0.035148,-0.036442,-0.022970,-0.022815,-0.024448,-0.015181,0.002692,-0.027139,NaN
2014-01-07,0.005773,0.013635,-0.003104,0.008655,0.007781,0.008327,0.004972,-0.000771,-0.000825,-0.002352,...,0.001976,-0.000814,0.000586,0.007361,-0.001723,-0.004671,-0.004126,-0.007989,0.004730,NaN
2014-01-08,-0.011954,0.048773,-0.000340,0.006444,0.003147,0.017815,-0.006431,0.015141,0.008112,-0.004680,...,-0.006365,0.003045,0.012179,0.011231,0.011629,-0.004728,0.006081,0.006364,0.001079,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-06-25,0.003783,-0.008935,-0.003832,-0.005251,0.008424,-0.013737,0.011497,0.005639,0.008205,-0.005481,...,0.000471,0.001531,-0.020913,0.009743,-0.023884,0.002773,0.007636,-0.012913,0.001366,0.007494
2024-06-26,0.004661,0.049481,0.001627,0.014013,0.009469,0.020820,0.009317,-0.004451,0.012099,0.006456,...,0.010504,0.028573,0.043841,0.015943,0.027637,0.013357,0.001447,0.011513,0.002147,0.028601
2024-06-27,-0.009856,-0.014990,-0.007472,-0.016877,-0.009377,-0.014311,-0.025626,-0.015710,-0.019958,-0.006014,...,-0.010848,-0.020059,-0.017633,-0.018320,-0.018150,-0.013592,0.006661,-0.019744,-0.006874,-0.016255
